In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

Defining RMSNorm Class :
Root Mean Square Layer Norm (no learned weights)

In [3]:
class RMSNorm(nn.Module):
  def __init__(self, d_model, eps: float = 1e-8):
    super().__init__()
    self.eps = eps

  def forward(self, x):
    # x: (batch, d_model)
    rms = torch.sqrt(x.pow(2).mean(dim=-1, keepdim=True)+ self.eps)
    return x/rms

Defining MTP class

In [4]:
class SimpleMTP(nn.Module):
  def __init__(self, d_model: int, vocab_size: int, num_heads: int, nhead: int = 2):
    super().__init__()
    self.d_model = d_model
    self.vocab_size = vocab_size
    self.num_heads = num_heads

    #Shared modules
    self.rmsnorm = RMSNorm(d_model)
    self.embed = nn.Embedding(vocab_size, d_model)
    self.unembed = nn.Linear(d_model, vocab_size, bias = False)

    #Share weights between embed and unembed
    self.unembed.weight = self.embed.weight

    #One Projection + one Transformer per head
    self.projections = nn.ModuleList([
        nn.Linear(2*d_model, d_model) for _ in range(num_heads)
    ])
    self.tranformers = nn.ModuleList([
        nn.TransformerEncoderLayer(d_model = d_model, nhead = nhead)
        for _ in range(num_heads)
    ])

  def forward(self, token_ids: torch.LongTensor, init_hidden: torch.Tensor = None):
    B, T = token_ids.shape
    device = token_ids.device

    # token embeddings: (B, T, d_model)
    embeds = self.embed(token_ids)

    # base hidden states
    if init_hidden is None:
      h0_seq = embeds
    else:
      h0_seq = init_hidden

    outputs = []
    # will hold (B, D, vocab_size) for each i
    # slide over positions where i + D < T
    max_i = T - self.num_heads - 1
    for i in range(0, max_i + 1):
      # previous hidden states for depth 0 at pos i
      h_prev = h0_seq[:, i, :]
      # collect logits for all k at this i
      logits_k = []
      for k in range(self.num_heads):
        #future token embed at pos i + (k+1)
        future_pos = i + (k+1)
        tok_embed = embeds[:, future_pos, :] #(B, d_model)

        #1)RMS normalise
        h_norm = self.rmsnorm(h_prev)
        e_norm = self.rmsnorm(tok_embed)

        #2) concatenate->(B, 2*d_model)
        merged = torch.cat([h_norm, e_norm], dim = -1)

        #3) project back to d_model
        proj = self.projections[k](merged)

        #4) Transformer block (expects shape (S, B, d_model))
        x = proj.unsqueeze(0)
        x = self.tranformers[k](x)
        h_curr = x.squeeze(0)

        #5) collect logits
        logits = self.unembed(h_curr)
        logits_k.append(logits)

        #6) chain hidden for next depth
        h_prev = h_curr
    #stack along depth axis -> (B, D, vocab_size)
      logits_k = torch.stack(logits_k, dim = 1)
      outputs.append(logits_k)
    #stack along sequence axis -> (T-D, B, D, V) then permute -> (B, T-D, D, V)
    out = torch.stack(outputs, dim = 0)
    out = out.permute(1, 0, 2, 3).contiguous()
    return out



pass input tokens through the model and generatee multiple next tokens.